# PHARVO-beta: Medicine Search Test

**Objective:** Verify that an authorized user can navigate to the **Medicines & Inventory** module, search for a medicine (`Paracetamol`), and view matching medicine records in the inventory table.

### Prerequisites
```bash
pip install selenium webdriver-manager
```
Ensure PHARVO frontend is running at `http://localhost:5173` and backend at `http://localhost:8000`.

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- Configuration & Test Credentials ---
BASE_URL = "http://localhost:5173"
USERNAME = "rafi"
PASSWORD = "787878"  # Replace with actual password
SEARCH_TERM = "Paracetamol"

# Step 1: Open browser and maximize window
driver = webdriver.Chrome()
driver.maximize_window()

# Set explicit wait helper (up to 10 seconds)
wait = WebDriverWait(driver, 10)

try:
    print(f"[INFO] Starting Medicine Search Test for '{SEARCH_TERM}'...")

    # Step 2: Open target PHARVO page and log in
    driver.get(f"{BASE_URL}/")

    # Step 3: Find login inputs
    username_field = wait.until(
        EC.visibility_of_element_located((By.ID, "username"))
    )
    password_field = wait.until(
        EC.visibility_of_element_located((By.ID, "password"))
    )

    # Step 4: Enter credentials
    username_field.clear()
    username_field.send_keys(USERNAME)

    password_field.clear()
    password_field.send_keys(PASSWORD)

    # Step 5: Perform login action
    sign_in_button = wait.until(
        EC.element_to_be_clickable((By.ID, "sign-in-btn"))
    )
    sign_in_button.click()

    # Step 6: Navigate to Medicines & Inventory module
    medicines_nav = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//aside//button[contains(., 'Medicines & Inventory')]")
        )
    )
    medicines_nav.click()

    # Verify Medicines & Inventory header is loaded
    page_header = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//header//h1[contains(text(), 'Medicines & Inventory')]")
        )
    )
    print(f"[INFO] Navigated to '{page_header.text}'.")

    # Step 7: Locate medicine search input field
    search_input = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//input[@aria-label='Search medicine or brand' or @placeholder='Search medicine or brand...']")
        )
    )

    # Step 8: Enter search query
    search_input.clear()
    search_input.send_keys(SEARCH_TERM)
    print(f"[INFO] Entered search query: '{SEARCH_TERM}'")

    # Step 9: Wait for filtered results in the inventory table
    # Wait for at least one table row to appear in the table body
    matching_cell = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, f"//table[contains(@class, 'med-table')]//tr[contains(@class, 'med-row-tr')]//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{SEARCH_TERM.lower()}')]")
        )
    )

    # Collect all visible result rows
    result_rows = driver.find_elements(By.XPATH, "//table[contains(@class, 'med-table')]//tbody//tr[contains(@class, 'med-row-tr')]")
    matching_count = len(result_rows)

    # Step 10: Verify the result and print PASS/FAIL information
    if matching_count > 0 and matching_cell.is_displayed():
        print(f"PASS: Medicine search successful for '{SEARCH_TERM}'.")
        print(f"      - Found {matching_count} matching medicine record(s).")
        print(f"      - First matched medicine: '{matching_cell.text}'")
    else:
        print(f"FAIL: No matching records found for '{SEARCH_TERM}'.")

except Exception as error:
    print(f"FAIL: Medicine search test encountered an error: {error}")

finally:
    # Step 11: Close browser
    print("[INFO] Cleaning up and closing browser...")
    driver.quit()
